In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/size_only/validation/ks/smooth_control'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# DeepRV Smooth GP Subspace Evaluation

For each vital rate and posterior lengthscale draw, this evaluation retains covariance directions whose smooth-kernel eigenvalues exceed the numerical jitter. The whitened coefficients are pooled across 50 independent DeepRV realizations and tested against the standard normal distribution. The figure shows the proportion of tests rejected at the 5% significance level across the posterior lengthscale distribution.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-white")

current_path = Path.cwd().resolve()
repo_root = PROJECT_ROOT
results_directory = GENERATED / 'soay/size_only/validation/ks/smooth_control'
if not results_directory.is_dir():
    raise FileNotFoundError(f"Results directory not found: {results_directory}")

parameter_results = pd.read_csv(results_directory / "ks_smooth_control_parameter_draw_results.csv")
figure_directory = results_directory / "figures"
figure_directory.mkdir(exist_ok=True)
print("Results Directory:", results_directory)

In [ ]:
vital_rate_order = ["Survival", "Growth", "Reproduction", "Recruit Size"]
colors = {"Survival": "#0072B2", "Growth": "#D55E00", "Reproduction": "#009E73", "Recruit Size": "#CC79A7"}

def style_axis(axis):
    axis.grid(False)
    axis.tick_params(labelsize=9, direction="out")
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(5.906, 5.2), sharex=True, sharey=True, constrained_layout=True)

for index, (axis, vital_rate) in enumerate(zip(axes.flat, vital_rate_order)):
    current = parameter_results[parameter_results["Vital Rate"] == vital_rate].copy()
    current["Lengthscale Bin"] = pd.qcut(current["Lengthscale"], q=30, duplicates="drop")
    binned = current.groupby("Lengthscale Bin", observed=True).agg(Lengthscale=("Lengthscale", "median"), Rejection_Proportion=("DeepRV Reject At 0.05", "mean"))

    axis.plot(binned["Lengthscale"], binned["Rejection_Proportion"], color=colors[vital_rate], marker="o", markersize=3, linewidth=1.2)
    axis.axhline(0.05, color="black", linestyle="--", linewidth=1.0)

    if index == 0:
        axis.text(0.97, 0.065, "5% Rate", transform=axis.get_yaxis_transform(), ha="right", va="bottom", fontsize=8, color="black")

    axis.set_ylim(-0.02, 1.02)
    axis.set_title(vital_rate, fontsize=11)
    style_axis(axis)

figure.supxlabel("Posterior Lengthscale", fontsize=11)
figure.supylabel("Rejection Proportion", fontsize=11)
#figure.savefig(figure_directory / "smooth_ks_rejection_across_lengthscales.pdf", bbox_inches="tight")
#figure.savefig(figure_directory / "smooth_ks_rejection_across_lengthscales.png", dpi=300, bbox_inches="tight")
plt.show()